# Experiments


### Project Setup

In [ ]:
!git clone https://github.com/stanfordnlp/stanza-train.git

In [ ]:
!pip install stanza

In [ ]:
#!source /content/stanza-train/config/config.sh
!source stanza-train/config/config.sh

In [ ]:
import os
#os.environ["UDBASE"] = "/content/stanza-train/data/udbase"
os.environ["UDBASE"] = "stanza-train/data/udbase"

In [ ]:
!pip install --upgrade torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install --upgrade transformers

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

In [ ]:
# remove the UD_English-TEST folder from the training data
import shutil

try:
    #shutil.rmtree("/content/stanza-train/data/udbase/UD_English-TEST")
    shutil.rmtree("stanza-train/data/udbase/UD_English-TEST")
    print(f"Directory and all its contents removed.")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
import time

### Model Training

In [ ]:
#UD_PATHS = [entry.name for entry in os.scandir("/content/stanza-train/data/udbase") if entry.is_dir() and entry.name.startswith("UD_")]
UD_PATHS = [entry.name for entry in os.scandir("stanza-train/data/udbase") if entry.is_dir() and entry.name.startswith("UD_")]

In [ ]:
UD_PATHS

In [ ]:
# create folder to hold all the logs from the commands we will be running
#os.makedirs("/content/logs", exist_ok=True)
os.makedirs("logs", exist_ok=True)

run tokenization for all languages

In [ ]:
for UD_PATH in UD_PATHS:
  print(f"Running tokenization for {UD_PATH}")
  start_time = time.perf_counter()
  # changed the logs file
  !python3 -m stanza.utils.datasets.prepare_tokenizer_treebank {UD_PATH} &> logs/{UD_PATH}_prepare_tokenizer_log.txt
  !python3 -m stanza.utils.training.run_tokenizer {UD_PATH} --steps 50 &> logs/{UD_PATH}_run_tokenizer_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

run pos for all languages

In [ ]:
for UD_PATH in UD_PATHS:
  print(f"Running pos for {UD_PATH}")
  start_time = time.perf_counter()
  # changed log path
  !python3 -m stanza.utils.datasets.prepare_pos_treebank {UD_PATH}  &> logs/{UD_PATH}_prepare_pos_log.txt
  !python3 -m stanza.utils.training.run_pos {UD_PATH} --no_pretrain --no_second_optim &> logs/{UD_PATH}_run_pos_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

run depparser for all languages

In [ ]:
# prepare a dummy pt file 
from pathlib import Path

Path("dummy.pt").write_bytes(b"")
print("Created empty dummy.pt successfully!")

In [ ]:
for UD_PATH in UD_PATHS:
  print(f"Running depparser for {UD_PATH}")
  start_time = time.perf_counter()
  # changed the logs destination
  !python3 -m stanza.utils.datasets.prepare_depparse_treebank {UD_PATH}  --wordvec_pretrain_file dummy.pt &> logs/{UD_PATH}_prepare_depparse_log.txt
  !python3 -m stanza.utils.training.run_depparse {UD_PATH} --no_pretrain &> logs/{UD_PATH}_run_depparse_log.txt
  end_time = time.perf_counter()
  total_time = end_time - start_time
  print(f"Total time taken: {total_time:.4f} seconds")

### Parsing

In [ ]:
import stanza
from stanza.utils.conll import CoNLL
import os
import shutil

language code taken from [stanza repo](https://github.com/stanfordnlp/stanza/blob/516b07140fdfdafdebc813ed8f3253097da5efd3/stanza/models/common/constant.py#L105)

In [ ]:
LANGUAGE_CODES = {
    "UD_Abaza": "abq",
    "UD_Abkhaz": "ab",
    "UD_Afrikaans": "af",
    "UD_Akkadian": "akk",
    "UD_Akuntsu": "aqz",
    "UD_Albanian": "sq",
    "UD_Alemannic": "gsw",
    "UD_Amharic": "am",
    "UD_Ancient_Greek": "grc",
    "UD_Ancient_Hebrew": "hbo",
    "UD_Apurina": "apu",
    "UD_Arabic": "ar",
    "UD_Armenian": "hy",
    "UD_Assyrian": "aii",
    "UD_Azerbaijani": "az",
    "UD_Bambara": "bm",
    "UD_Basque": "eu",
    "UD_Bavarian": "bar",
    "UD_Beja": "bej",
    "UD_Belarusian": "be",
    "UD_Bengali": "bn",
    "UD_Bhojpuri": "bho",
    "UD_Bokota": "sab",
    "UD_Bororo": "bor",
    "UD_Breton": "br",
    "UD_Bulgarian": "bg",
    "UD_Buryat": "bxr",
    "UD_Cantonese": "yue",
    "UD_Cappadocian": "cpg",
    "UD_Catalan": "ca",
    "UD_Cebuano": "ceb",
    "UD_Central_Kurdish": "ckb",
    "UD_Chinese": "zh",
    "UD_Chintang": "ctn",
    "UD_Chukchi": "ckt",
    "UD_Classical_Armenian": "xcl",
    "UD_Classical_Chinese": "lzh",
    "UD_Coptic": "cop",
    "UD_Croatian": "hr",
    "UD_Czech": "cs",
    "UD_Danish": "da",
    "UD_Dutch": "nl",
    "UD_Egyptian": "egy",
    "UD_English": "en",
    "UD_Erzya": "myv",
    "UD_Esperanto": "eo",
    "UD_Estonian": "et",
    "UD_Faroese": "fo",
    "UD_Finnish": "fi",
    "UD_French": "fr",
    "UD_Frisian_Dutch": "qfn",
    "UD_Galician": "gl",
    "UD_Georgian": "ka",
    "UD_German": "de",
    "UD_Gheg": "aln",
    "UD_Gothic": "got",
    "UD_Greek": "el",
    "UD_Guajajara": "gub",
    "UD_Guarani": "gn",
    "UD_Gujarati": "gu",
    "UD_Gwichin": "gwi",
    "UD_Haitian_Creole": "ht",
    "UD_Hausa": "ha",
    "UD_Hebrew": "he",
    "UD_Highland_Puebla_Nahuatl": "azz",
    "UD_Hindi": "hi",
    "UD_Hittite": "hit",
    "UD_Hungarian": "hu",
    "UD_Icelandic": "is",
    "UD_Ika": "arh",
    "UD_Indonesian": "id",
    "UD_Irish": "ga",
    "UD_Italian": "it",
    "UD_Japanese": "ja",
    "UD_Javanese": "jv",
    "UD_Kaapor": "urb",
    "UD_Kangri": "xnr",
    "UD_Karelian": "krl",
    "UD_Karo": "arr",
    "UD_Kazakh": "kk",
    "UD_Khoekhoe": "naq",
    "UD_Khunsari": "kfm",
    "UD_Kiche": "quc",
    "UD_Komi_Permyak": "koi",
    "UD_Komi_Zyrian": "kpv",
    "UD_Korean": "ko",
    "UD_Kyrgyz": "ky",
    "UD_Latgalian": "ltg",
    "UD_Latin": "la",
    "UD_Latvian": "lv",
    "UD_Ligurian": "lij",
    "UD_Lithuanian": "lt",
    "UD_Livvi": "olo",
    "UD_Low_Saxon": "nds",
    "UD_Luxembourgish": "lb",
    "UD_Macedonian": "mk",
    "UD_Madi": "jaa",
    "UD_Maghrebi_Arabic_French": "qaf",
    "UD_Makurap": "mpu",
    "UD_Malayalam": "ml",
    "UD_Maltese": "mt",
    "UD_Manx": "gv",
    "UD_Marathi": "mr",
    "UD_Mbya_Guarani": "gun",
    "UD_Middle_French": "frm",
    "UD_Moksha": "mdf",
    "UD_Munduruku": "myu",
    "UD_Naga": "nmf",
    "UD_Naija": "pcm",
    "UD_Nayini": "nyq",
    "UD_Neapolitan": "nap",
    "UD_Nenets": "yrk",
    "UD_Nheengatu": "yrl",
    "UD_North_Sami": "sme",
    "UD_Northern_Kurdish": "kmr",
    "UD_Northwest_Gbaya": "gya",
    "UD_Norwegian": "no",
    "UD_Occitan": "oc",
    "UD_Odia": "or",
    "UD_Old_Church_Slavonic": "cu",
    "UD_Old_East_Slavic": "orv",
    "UD_Old_English": "ang",
    "UD_Old_French": "fro",
    "UD_Old_Irish": "sga",
    "UD_Old_Occitan": "pro",
    "UD_Old_Turkish": "otk",
    "UD_Ottoman_Turkish": "ota",
    "UD_Pashto": "ps",
    "UD_Paumari": "pad",
    "UD_Persian": "fa",
    "UD_Pesh": "pay",
    "UD_Phrygian": "xpg",
    "UD_Polish": "pl",
    "UD_Pomak": "qpm",
    "UD_Portuguese": "pt",
    "UD_Romanian": "ro",
    "UD_Russian": "ru",
    "UD_Sanskrit": "sa",
    "UD_Scottish_Gaelic": "gd",
    "UD_Serbian": "sr",
    "UD_Shanghainese": "wuu",
    "UD_Sicilian": "scn",
    "UD_Sindhi": "sd",
    "UD_Sinhala": "si",
    "UD_Skolt_Sami": "sms",
    "UD_Slovak": "sk",
    "UD_Slovenian": "sl",
    "UD_Soi": "soj",
    "UD_South_Levantine_Arabic": "ajp",
    "UD_Southern_Kurdish": "sdh",
    "UD_Spanish_Sign_Language": "ssp",
    "UD_Spanish": "es",
    "UD_Swedish_Sign_Language": "swl",
    "UD_Swedish": "sv",
    "UD_Tagalog": "tl",
    "UD_Tamil": "ta",
    "UD_Tatar": "tt",
    "UD_Teko": "eme",
    "UD_Telugu_English": "qte",
    "UD_Telugu": "te",
    "UD_Thai": "th",
    "UD_Tswana": "tn",
    "UD_Tupinamba": "tpn",
    "UD_Turkish_English": "qti",
    "UD_Turkish_German": "qtd",
    "UD_Turkish": "tr",
    "UD_Ukrainian": "uk",
    "UD_Umbrian": "xum",
    "UD_Upper_Sorbian": "hsb",
    "UD_Urdu": "ur",
    "UD_Uyghur": "ug",
    "UD_Uzbek": "uz",
    "UD_Veps": "vep",
    "UD_Vietnamese": "vi",
    "UD_Warlpiri": "wbp",
    "UD_Welsh": "cy",
    "UD_Western_Armenian": "hyw",
    "UD_Western_Sierra_Puebla_Nahuatl": "nhi",
    "UD_Wolof": "wo",
    "UD_Xavante": "xav",
    "UD_Xibe": "sjo",
    "UD_Yakut": "sah",
    "UD_Yiddish": "yi",
    "UD_Yoruba": "yo",
    "UD_Yupik": "ess",
    "UD_Zaar": "say",
}

In [ ]:
def get_model_name(ud_path_name):
  language_code = LANGUAGE_CODES[ud_path_name.split("-")[0]]
  extra_arguments = ud_path_name.split("-")[1]
  return "_".join([language_code, extra_arguments.lower()])

In [ ]:
def get_first_pt_file(directory):
    if os.path.exists(directory):
        files = [f for f in os.listdir(directory) if f.endswith('.pt')]
        if files:
            return os.path.join(directory, files[0])
    return None

In [ ]:
# changed paths
for UD_PATH in UD_PATHS:
  model_name = get_model_name(UD_PATH)
  lang = model_name.split("_")[0]
  # only for chinese the name conventions do not hold
  if lang == "zh":
     model_name = "zh-hans_gsdsimp"
  

  print(f"Working on model for {UD_PATH}")

  #f_charlm_dir = f"/root/stanza_resources/{lang}/forward_charlm"
  #b_charlm_dir = f"/root/stanza_resources/{lang}/backward_charlm"
  f_charlm_dir = f"/home/beauty/stanza_resources/{lang}/forward_charlm"
  b_charlm_dir = f"/home/beauty/stanza_resources/{lang}/backward_charlm"

  forward_charlm_path = get_first_pt_file(f_charlm_dir)
  backward_charlm_path = get_first_pt_file(b_charlm_dir)

  print(f"Found Forward CharLM: {forward_charlm_path}")
  print(f"Found Backward CharLM: {backward_charlm_path}")

  if os.path.exists(f'saved_models/tokenize/{model_name}_charlm_tokenizer.pt'):
    tokenize_model_path = f'saved_models/tokenize/{model_name}_charlm_tokenizer.pt'
  elif os.path.exists(f'saved_models/tokenize/{model_name}_nocharlm_tokenizer.pt'):
    tokenize_model_path = f'saved_models/tokenize/{model_name}_nocharlm_tokenizer.pt'
  else:
    raise FileNotFoundError("No tokenizer found")

  # changed path and name of the model
  if os.path.exists(f'saved_models/pos/{model_name}_charlm_tagger.pt'):
    pos_model_path = f'saved_models/pos/{model_name}_charlm_tagger.pt'
  elif os.path.exists(f'saved_models/pos/{model_name}_nocharlm_tagger.pt'):
     pos_model_path = f'saved_models/pos/{model_name}_nocharlm_tagger.pt'
  elif os.path.exists(f'saved_models/pos/{model_name}_nopretrain_tagger.pt'):
      pos_model_path = f'saved_models/pos/{model_name}_nopretrain_tagger.pt'
  else:
    raise FileNotFoundError("No pos tagger found")

  if os.path.exists(f'saved_models/depparse/{model_name}_charlm_parser.pt'):
    depparse_model_path = f'saved_models/depparse/{model_name}_charlm_parser.pt'
  elif os.path.exists(f'saved_models/depparse/{model_name}_nocharlm_parser.pt'):
    depparse_model_path = f'saved_models/depparse/{model_name}_nocharlm_parser.pt'
  elif os.path.exists(f"saved_models/depparse/{model_name}_nopretrain_parser.pt"):
    depparse_model_path = f"saved_models/depparse/{model_name}_nopretrain_parser.pt"
  else:
    raise FileNotFoundError("No parser found")

  processors = "tokenize,pos,lemma,depparse"

  pipeline_config = {
      "lang": lang,
      "use_gpu": True,
      "package": None,
      "download_method": None,
      "processors": processors,
      "tokenize_model_path": tokenize_model_path,
      "pos_model_path": pos_model_path,
      "depparse_model_path": depparse_model_path,
      "depparse_pretrain_path": "dummy.pt",
      "tokenize_pretokenized": True,
      "lemma_use_identity": True,
      "allow_unknown_language": True,
  }

  if forward_charlm_path:
      pipeline_config["tokenize_forward_charlm_path"] = forward_charlm_path

  if backward_charlm_path:
      pipeline_config["tokenize_backward_charlm_path"] = backward_charlm_path


  nlp = stanza.Pipeline(**pipeline_config)

  for filename in os.listdir("test_data"):
      print(f'Working on file {filename}')
      file_path = os.path.join("test_data", filename)

      doc = CoNLL.conll2doc(file_path)
      doc = nlp(doc)
      CoNLL.write_doc2conll(doc, f"parsed_data/{model_name}-on-{filename.split('.')[0]}.conllu")